# GenFragility Graph — Quick Tour

This notebook walks through the core query patterns supported by the bundled
`GraphAPI`. Inside the Docker image, the graph and QID index are already on
disk at `/app/data/`; outside the image, point `GENFRAG_GRAPH_PATH` /
`GENFRAG_QID_INDEX_PATH` at your own copies.

In [ ]:
import os, sys
sys.path.insert(0, '/app')
from graph_api import GraphAPI

GRAPH = os.environ.get('GENFRAG_GRAPH_PATH', '/app/data/final.pkl')
QID   = os.environ.get('GENFRAG_QID_INDEX_PATH', '/app/data/graph_qid_index.json')

api = GraphAPI(GRAPH, QID)
G   = api.graph
print(api.banner())

## 1. Look up in-degree and popularity

Popularity is QID-aggregated in-degree (paper §5.1) — `"USA"` and `"United States"`
collapse into a single `Q30` count.

In [ ]:
for key in ['Brooklyn', 'Q30', 'united states', 'NotAnEntity']:
    print(key, '->', api.get_popularity(key))

## 2. Search by substring (case-insensitive)

In [ ]:
api.search('jay-z', limit=10)

## 3. Top-N hubs by aggregated in-degree

In [ ]:
for row in api.top_hubs(15, by='qid'):
    print(f"{row['popularity']:>6}  {row['qid']:<8}  {row['canonical']!r}")

## 4. Per-node detail: degrees + relation histogram + sample questions

In [ ]:
api.node_info('United States')

## 5. Drop down to raw networkx for anything else

`G` is a plain `networkx.DiGraph`, so all of the networkx algorithm zoo
(PageRank, betweenness, BFS, shortest paths, ...) works directly.

In [ ]:
import networkx as nx
# PageRank can be slow on 100k nodes; tune max_iter / tol if needed.
pr = nx.pagerank(G, alpha=0.85, max_iter=50)
top_pr = sorted(pr.items(), key=lambda kv: kv[1], reverse=True)[:10]
for node, score in top_pr:
    print(f"{score:.5f}  {node}")